In [38]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import io
from PIL import Image as PILImage
import joblib
from skimage.feature import hog  # <--- CHANGED: Import HOG

# --- 1. SETUP HELPERS ---

# <--- CHANGED: Removed LocalBinaryPatterns Class (Not needed for HOG)

def extract_hog_features(image):
    """
    Extracts HOG features. Must match the training configuration exactly.
    """
    features = hog(image, 
                   orientations=9, 
                   pixels_per_cell=(8, 8), 
                   cells_per_block=(2, 2), 
                   block_norm='L2-Hys', 
                   visualize=False)
    return features

# Constants
id_to_name = { 1: "Besheer", 2: "Ashraf", 3: "Seif", 4: "Sallam", 5: "Roger", 6: "Omar" }
# <--- CHANGED: Removed 'desc = LocalBinaryPatterns(...)'

# --- 2. LOAD MODELS ---
models = {}
try:
    # LBPH (OpenCV Native) handles its own features internally
    lbph = cv2.face.LBPHFaceRecognizer_create(radius=1, neighbors=8, grid_x=8, grid_y=8)
    lbph.read('trainer.yml')
    models['LBPH'] = lbph
    print("LBPH loaded.")
except: print("LBPH not found.")

try:
    svm = joblib.load('svm_face_model.pkl')
    models['SVM'] = svm
    print("SVM loaded.")
except: print("SVM not found.")

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml')

# --- 3. GUI ELEMENTS ---
model_selector = widgets.ToggleButtons(
    options=['LBPH', 'SVM'],
    description='Model:',
    button_style='info'
)

uploader = widgets.FileUpload(accept='image/*', multiple=False)
out_image = widgets.Output()
out_results = widgets.Output()

def process_image(change):
    out_image.clear_output()
    out_results.clear_output()
    
    if not uploader.value: return
    
    # Image Loading
    if isinstance(uploader.value, tuple): f = uploader.value[0]
    else: f = next(iter(uploader.value.values()))
    content = f['content']
    if isinstance(content, memoryview): content = content.tobytes()
    
    img_pil = PILImage.open(io.BytesIO(content))
    img_np = np.array(img_pil)
    
    if len(img_np.shape) == 3:
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    else:
        img_bgr = img_np.copy()
        gray = img_np

    # --- INTELLIGENT DETECTION SWITCH ---
    current_model = model_selector.value
    
    if current_model == 'SVM':
        # STRICT MODE for SVM
        detect_neighbors = 4
    else:
        # LOOSE MODE for LBPH
        detect_neighbors = 3

    faces = face_cascade.detectMultiScale(
        gray, 
        scaleFactor=1.06, 
        minNeighbors=detect_neighbors, 
        minSize=(49, 49)
    )
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    candidates = []

    for i, (x, y, w, h) in enumerate(faces):
        # ROI
        face_roi = gray[y:y+h, x:x+w]
        
        # RESIZE MUST MATCH TRAINING (200x200)
        face_resized = cv2.resize(face_roi, (200, 200), interpolation=cv2.INTER_CUBIC)
        face_smooth = cv2.bilateralFilter(face_resized, 5, 75, 75)
        face_final = clahe.apply(face_smooth)
        
        name = "Unknown"
        display_score = 0.0
        
        # --- SVM PREDICTION ---
        if current_model == 'SVM':
            # <--- CHANGED: Use HOG extractor instead of LBP descriptor
            hog_features = extract_hog_features(face_final)
            
            # Reshape for single sample prediction
            # SVM expects 2D array: (1, n_features)
            proba = models['SVM'].predict_proba([hog_features])[0]
            idx = np.argmax(proba)
            score = proba[idx]
            
            display_score = score * 100
            
            # SVM Threshold
            if score > 0.35: 
                pred_id = models['SVM'].classes_[idx]
                name = id_to_name.get(pred_id, "Unknown")
            
        # --- LBPH PREDICTION ---
        else:
            pred_id, dist = models['LBPH'].predict(face_final)
            display_score = dist
            
            # LBPH Strict Discard Logic
            if dist > 80: continue 
            
            name = id_to_name.get(pred_id, "Unknown")

        candidates.append({
            'id': i, 'bbox': (x, y, w, h),
            'name': name, 'score': display_score,
            'model': current_model, 'original_roi': img_np[y:y+h, x:x+w]
        })

    # --- SORT & DEDUPLICATE ---
    reverse_sort = (current_model == 'SVM')
    candidates.sort(key=lambda x: x['score'], reverse=reverse_sort)
    
    final_preds = []
    seen = set()
    
    for cand in candidates:
        n = cand['name']
        if n in seen and n != "Unknown":
            cand['is_dup'] = True
        else:
            if n != "Unknown": seen.add(n)
            cand['is_dup'] = False
        final_preds.append(cand)
        
    final_preds.sort(key=lambda x: x['id']) 

    # --- DISPLAY ---
    with out_image:
        img_disp = img_np.copy()
        if not final_preds: print(f"No faces detected (Mode: {current_model}).")
        
        for pred in final_preds:
            x, y, w, h = pred['bbox']
            name = pred['name']
            score = pred['score']
            is_dup = pred['is_dup']
            
            if name == "Unknown": color = (255, 0, 0)
            elif is_dup: color = (0, 165, 255)
            else: color = (0, 255, 0)
            
            cv2.rectangle(img_disp, (x, y), (x+w, y+h), color, 2)
            
            label_val = f"{int(score)}%" if pred['model'] == 'SVM' else f"C:{int(score)}"
            txt = f"{name} ({label_val})" + ("*" if is_dup else "")
            
            cv2.putText(img_disp, txt, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            
        display(PILImage.fromarray(img_disp))

    # --- TABLE ---
    with out_results:
        if not final_preds: return
        rows = []
        for pred in final_preds:
            roi = cv2.cvtColor(pred['original_roi'], cv2.COLOR_RGB2BGR)
            thumb = widgets.Image(value=cv2.imencode('.png', roi)[1].tobytes(), format='png', width=80)
            
            if pred['model'] == 'SVM': score_txt = f"Conf: {pred['score']:.1f}%"
            else: score_txt = f"Dist: {pred['score']:.1f}"
            
            dup = " <b style='color:orange'>(Dup)</b>" if pred['is_dup'] else ""
            info = widgets.HTML(f"<b>{pred['name']}</b>{dup}<br>{score_txt}")
            rows.append(widgets.HBox([thumb, info]))
        display(widgets.VBox(rows))

# Observers
uploader.observe(process_image, names='value')
model_selector.observe(process_image, names='value')

display(widgets.VBox([
    widgets.HTML("<h2>Multi-Model Face Recognition (Auto-Strict SVM)</h2>"),
    widgets.HBox([model_selector]),
    uploader, 
    out_image, 
    out_results
]))

LBPH loaded.
SVM loaded.
